# k-Means Clustering

### YOUR NAME HERE

Here we will explore a class of unsupervised machine learning models: clustering algorithms.
Clustering algorithms seek to learn, from the properties of the data, an optimal division or discrete labeling of groups of points.

Many clustering algorithms are available in Scikit-Learn and elsewhere, but perhaps the simplest to understand is an algorithm known as *k-means clustering*, which is implemented in ``sklearn.cluster.KMeans``.

We begin with some imports:

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()  # for plot styling
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import itertools
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn import preprocessing
from sklearn.decomposition import PCA

## Excercise 1: 
Import the Candy data and show the first 10 rows.  Write a dataset description based on the information provided.  

## Dataset Description
TBD

## Exercise 2:

Create a scatter plot comparing the price percent and win percent.  Write a few sentences in the reflection below that describes what you observe (are there any obvious groups?).  

### Reflection
TODO

## Exercise 3: 
In the following cell, convert the dataframe into a 2D numpy array called X that only contains numerical data (by dropping the one categorical column "competitorname" - this column does not contain relevant info for clustering). 

The following code will create a kMeans classifier using the data in ``X``.  Run this cell, and feel free to come back to experiment with different values of k.  You should not need to modify this cell if you completed the previous cell as expected.  

In [ ]:
kmeans = KMeans(n_clusters=2)
kmeans.fit(X)
y_kmeans = kmeans.predict(X)

Let's visualize the results by plotting the data colored by these labels using the columns `pricepercent` and `winpercent`.
We will also plot the cluster centers in yellow as determined by the *k*-means estimator:

In [ ]:
plt.scatter(X[:, 10], X[:, 11], c=y_kmeans, s=50)

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 10], centers[:, 11], c='blue', s=200, alpha=0.5);

## Exercise 4: 
This clustering isn't necessarily what you would expect, right?  This is most likely due to scaling bias in the distance calculation.  To address this, we need to scale the axes.  In the following cell, scale values using sklearn.preprocessing.scale(). You should re-do the clustering and plot the results to compare them to what you generated earlier. 

**Note: All further clustering should be done on the scaled data**

### Reflection:

How do these results compare with your unscaled results?

## Exercise 5: 
Try (and display graphs of) three additional k values to see how KMeans clusters the data.  Additionally, apply Principal Components Analysis (PCA) to give a "good" view of your 11-dimensional data in 2 dimensions (see the unsupervised learning notebook from lecture for an exmample on how to do this).  Document your experiments and write a short description in the reflection section below - which set of k-means clusters appeared to give the most distinct separation between groups?

### Reflection
TODO

## Excercise 6:

One way discussed in class to evaluate how "good" one group of clusters is is to compare the Davies-Bouldin score. Compute and display the Davies-Bouldin score for the 4 values of 'k' that you used above. Be sure to clearly label which output values belong to which graphs. 

Does the 'k' that the Davies Bouldin score denotes as creating the best clusters align with your "eye-balled" analysis from Exercise 5? If not, why might this be?

### Reflection

TODO

## Exercise 7

Another way to evaluate which number of clusters is best is to use the "within cluster sum of squares" or inertia.  It is a measure of how coherent the clusters are.  We can look at a variety of k-values to see how the inertia changes for different k-values.  This is not a perfect measure (it is subject to several types of biases), but it can be used to compare different k-values.  

To use the "Elbow Method" you want to find an "elbow" where there is a distinct change in the plot of different k-values.  It is not always very obvious, but this describes what to look for: https://www.geeksforgeeks.org/elbow-method-for-optimal-value-of-k-in-kmeans/ 

Calculate the "inertia" for k-means clusters with k varying between 2 and 6 and display and "elbow plot". Using the "elbow" method, which number of clusters would you say is best? Why? (put answer in reflection below).

Note that for each clustering object you create, it has an attribute called "inertia_" that represents this value. 

## Reflection
TODO

## Exercise 8

A third metric used to help identify how many clusters you should have is called the Silhouette Coefficient. You can read up on it in the following locations: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html and https://towardsdatascience.com/silhouette-coefficient-validating-clustering-techniques-e976bb81d10c.

Calculate and display the Silhouette Coefficient for k-means clusters with k varying between 2 and 6. Which k-value does the Silhouette Coefficient claim creates the best clusters?

**Answer here:**

# Example: k-means on images

To start, let's take a look at applying *k*-means on simple handwritten digits images.  Here we will attempt to use *k*-means to try to identify similar digits *without using the original label information*.  We will start by loading the digits and then finding the ``KMeans`` clusters.

The digits data consist of 1,797 samples with 64 features, where each of the 64 features is the brightness of one pixel in an 8×8 image.  We start by loading the data

In [1]:
from sklearn.datasets import load_digits
digits = load_digits()

Let's take a look at one of the images.  

In [ ]:
plt.imshow(digits.images[0], cmap='gray')
plt.show()

## Exercise 9:
Write a few lines of code to create a KMeans model using ``digits.data`` and 10 clusters.  No need to plot, just create the clusters for now. Note that you should name your model object `kmeans` and the assigned clusters to `clusters` so that the following code cells will run appropriately.

Let's see what the cluster centers look like:

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(8, 3))
centers = kmeans.cluster_centers_.reshape(10, 8, 8)
for axi, center in zip(ax.flat, centers):
    axi.set(xticks=[], yticks=[])
    axi.imshow(center, interpolation='nearest', cmap=plt.cm.binary)

We see that *even without the labels*, ``KMeans`` is able to cluster whose centers are recognizable digits, with the exception of 1 and 8.  

*k*-means knows nothing about the identity of the cluster, so the clusters are not in any specific order.  To identify the actual order, we can use the labels in the following code.  

In [ ]:
from scipy.stats import mode

labels = np.zeros_like(clusters)
for i in range(10):
    mask = (clusters == i) #array of booleans for each cluster group
    labels[mask] = mode(digits.target[mask])[0] #get "mode" of each cluster group's label

Now we can check how accurate our unsupervised clustering was in finding similar digits within the data:

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(digits.target, labels)

With just a simple *k*-means algorithm, we discovered the correct grouping for 80% of the input digits!
Let's check the confusion matrix for this:

In [ ]:
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(digits.target, labels)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=digits.target_names,
            yticklabels=digits.target_names)
plt.xlabel('true label')
plt.ylabel('predicted label')

As we might expect from the cluster centers we visualized before, the main point of confusion is between the eights and ones.
But this still shows that using *k*-means, we can essentially build a digit classifier *without reference to any known labels*!

## Exercise 10:
Let's try this with higher resolution images!  Use the code below to import the 28x28 DIGITS images. Note, you will either need to be on Rosie's Tensorflow 2.x container, or you will likely need to install some libraries locally.

Then complete the following steps:
1. Plot an example image 
2. Train a KMeans cluster using the testing data.  Note that you will need to reshape the data to be in the correct format. Note that the N images will result in an N x 28 x 28 array. You will need to reshape the data to be an N x (28\*28) array. Look into the numpy .reshape() method to accomplish this.
3. Plot the cluster centers
4. Calculate the accuracy
5. Plot the confusion matrix
6. Use the training data and re-run the experiment to see how sensitive the model is to the data.  
7. Write a short reflection on what you observed from this and why it might be the case.  

In [ ]:
import tensorflow as tf
from tensorflow import keras
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

### Reflection
TODO